# 4-model Ensemble Inference (speed measurement)

## Models (all OpenVINO IR, internet=off, submit-ready)

| Model | Backbone | Window | Mel | Format | Weight |
|---|---|---|---|---|---|
| exp017 R2 | eca_nfnet_l0 | 5s simple | Tucker (256/2048/512) | OV | 0.30 |
| exp081 R2 | eca_nfnet_l0 | 10s sliding (pad 2.5s) | Tucker | OV | 0.25 |
| exp082 R2 | eca_nfnet_l0 | 20s sliding (pad 7.5s) | Babych (224/4096/1252) | OV | 0.15 |
| exp015 R2 | convnext_pico | 5s simple | Tucker | OV | 0.30 |

**Weight sum = 1.00**

## Pipeline (working v6 pattern)
1. Per model: forward → clip_logits + framewise → **LOGIT space blend** (0.5/0.5)
2. Cross-model: weighted sum in LOGIT space
3. Smooth across 12 windows with [0.1, 0.2, 0.4, 0.2, 0.1]
4. **Sigmoid LAST**

## Window handling

| window_type | Logic | Pad spec |
|---|---|---|
| `simple` (5s) | 60s reshape → (12, 5s) | no padding |
| `sliding` (10s) | padded 65s → step 5s → 12 windows of 10s | 80000 samples = 2.5s |
| `sliding` (20s) | padded 75s → step 5s → 12 windows of 20s | 240000 samples = 7.5s |

## Speed monitoring
- Per-50-files progress + ETA
- Per-model wall-clock breakdown
- Final total time

## Inputs (4 weights dataset + openvino wheel)
- `birdclef-2026` (competition)
- `cooolz/openvino-package` (OV wheel for offline install)
- `maekeso/birdclef2026-exp015-weights` (OV r2_ov/)
- `maekeso/birdclef2026-exp017-weights` (OV r2_ov/)
- `maekeso/birdclef2026-exp081-weights` (OV r2_ov/)
- `maekeso/birdclef2026-exp082-weights` (OV r2_ov/)

## Expected timing
- 4 OV models on Kaggle CPU
- ~15-30 min per 700 file subset (vs 90 min 6-model fail)
- Full rerun (~1500 files): ~30-60 min (within 90 min limit)


In [ ]:
# ============================================================
# Cell 1: Setup — install openvino from wheel (internet=off submit-ready)
# ============================================================
# ★ openvino wheel from cooolz/openvino-package Dataset (cp312、internet=off OK)
import os, sys
OV_WHEEL_DIR = '/kaggle/input/datasets/cooolz/openvino-package'
if not os.path.exists(OV_WHEEL_DIR):
    OV_WHEEL_DIR = '/kaggle/input/openvino-package'   # fallback
assert os.path.exists(OV_WHEEL_DIR), f'openvino wheel dataset not attached: {OV_WHEEL_DIR}'
import glob
ov_whl = glob.glob(f'{OV_WHEEL_DIR}/openvino-*.whl')
tel_whl = glob.glob(f'{OV_WHEEL_DIR}/openvino_telemetry-*.whl')
assert ov_whl and tel_whl, f'missing wheels in {OV_WHEEL_DIR}'
print(f'Installing from wheels: {[w.split("/")[-1] for w in ov_whl + tel_whl]}')

# --no-deps to avoid numpy upgrade (which breaks scipy)
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                       *ov_whl, *tel_whl])
print('openvino wheel installed')

import time, json, gc, glob as glob_mod
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from scipy.ndimage import convolve1d
import soundfile as sf
import librosa
import openvino as ov
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cpu')
torch.set_num_threads(4)
print(f'torch={torch.__version__}, timm={timm.__version__}, ov={ov.__version__}')
print(f'numpy={np.__version__}, device={device}, num_threads={torch.get_num_threads()}')

GLOBAL_START = time.time()


In [ ]:
# ============================================================
# Cell 2: Config — mel specs + model paths + ensemble weights
# ============================================================
SR = 32000
N_CLASSES = 234
PERCH_EMBED_DIM = 1536
USE_PERCH_DISTILL = True

# Paths
BASE = None
for p in [Path('/kaggle/input/competitions/birdclef-2026'),
          Path('/kaggle/input/birdclef-2026')]:
    if p.exists():
        BASE = p; break
assert BASE is not None
TEST_DIR = BASE / 'test_soundscapes'
SAMPLE_SUB = pd.read_csv(BASE / 'sample_submission.csv')
PRIMARY_LABELS = SAMPLE_SUB.columns[1:].tolist()
assert len(PRIMARY_LABELS) == N_CLASSES

# Mel transform configs
MEL_CONFIGS = {
    'Tucker':  dict(n_mels=256, n_fft=2048, hop=512,  win=2048, fmin=20, fmax=16000),
    'Babych':  dict(n_mels=224, n_fft=4096, hop=1252, win=4096, fmin=0,  fmax=16000),
    'Hi-freq': dict(n_mels=256, n_fft=4096, hop=512,  win=4096, fmin=20, fmax=16000),
    'Hi-time': dict(n_mels=256, n_fft=2048, hop=256,  win=2048, fmin=20, fmax=16000),
}

# Model configs
MODELS_CFG = {
    'exp017_r2': {
        'dataset_slug': 'birdclef2026-exp017-weights',
        'mel': 'Tucker', 'duration': 5, 'backbone': 'eca_nfnet_l0',
        'window_type': 'simple',   # simple 12 chunks of 5s
        'format': 'ov', 'ov_path': 'r2_ov/model.xml',
        'pytorch_ckpt': 'r2_ckpt_best_ns22.pth',
        'weight': 0.30,
    },
    'exp081_r2': {
        'dataset_slug': 'birdclef2026-exp081-weights',
        'mel': 'Tucker', 'duration': 10, 'backbone': 'eca_nfnet_l0',
        'window_type': 'sliding',  # padded 65s, 12 windows of 10s step 5s
        'format': 'ov', 'ov_path': 'r2_ov/model.xml',
        'pytorch_ckpt': 'r2/ckpt_best_ns22.pth',
        'weight': 0.25,
    },
    'exp082_r2': {
        'dataset_slug': 'birdclef2026-exp082-weights',
        'mel': 'Babych', 'duration': 20, 'backbone': 'eca_nfnet_l0',
        'window_type': 'sliding',  # padded 75s, 12 windows of 20s step 5s
        'format': 'ov', 'ov_path': 'r2_ov/model.xml',
        'pytorch_ckpt': 'r2/ckpt_best_ns22.pth',
        'weight': 0.15,
    },
    'exp015_r2': {
        'dataset_slug': 'birdclef2026-exp015-weights',
        'mel': 'Tucker', 'duration': 5, 'backbone': 'convnext_pico.d1_in1k',
        'window_type': 'simple',
        'format': 'ov', 'ov_path': 'r2_ov/model.xml',
        'pytorch_ckpt': 'r2_ckpt_best_ns22.pth',
        'weight': 0.30,
    },
}

# Verify weights sum to 1.0
wsum = sum(c['weight'] for c in MODELS_CFG.values())
assert abs(wsum - 1.0) < 1e-6, f'weights sum to {wsum}, not 1.0'
print(f'Models: {list(MODELS_CFG.keys())}')
print(f'Weights sum: {wsum:.3f}')

OUT_DIR = Path('/kaggle/working')


In [ ]:
# ============================================================
# Cell 3: BirdSEDModel class for PyTorch CPU fallback (exp083 R1, exp084 R1)
# ============================================================
class GeMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
    def forward(self, feature_map):
        return self.proj(feature_map.mean(dim=[2, 3]))


class BirdSEDModel(nn.Module):
    def __init__(self, backbone_name, n_mels, time_frames,
                 num_classes=N_CLASSES, drop_path_rate=0.0, hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=False, in_chans=1,
            num_classes=0, global_pool='', drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            dummy = torch.randn(1, 1, n_mels, time_frames)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25), nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True), nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False):
        h = self.backbone(x)
        h_cls = self.gem_freq(h)
        h_cls = h_cls.permute(0, 2, 1)
        h_cls = self.dense(h_cls)
        h_cls = h_cls.permute(0, 2, 1)
        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise, dim=2)
        if return_framewise:
            return clip_logits, framewise.permute(0, 2, 1)
        return clip_logits

print('OK BirdSEDModel ready')


In [ ]:
# ============================================================
# Cell 4: Mel transforms (4 configs)
# ============================================================
class MelSpecTransform(nn.Module):
    def __init__(self, n_fft, hop, win, n_mels, fmin, fmax):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=n_fft, hop_length=hop, win_length=win,
            n_mels=n_mels, f_min=fmin, f_max=fmax, power=2.0,
        )
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, wav):
        return self.db(self.mel(wav))


MEL_TRANSFORMS = {}
for name, cfg in MEL_CONFIGS.items():
    MEL_TRANSFORMS[name] = MelSpecTransform(
        n_fft=cfg['n_fft'], hop=cfg['hop'], win=cfg['win'],
        n_mels=cfg['n_mels'], fmin=cfg['fmin'], fmax=cfg['fmax'],
    ).to(device)
    print(f'  {name}: n_fft={cfg["n_fft"]}, hop={cfg["hop"]}, n_mels={cfg["n_mels"]}')

def normalize_mel(mel):
    '''Per-sample normalize.'''
    for i in range(mel.size(0)):
        mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
    return mel

print('OK mel transforms ready')


In [ ]:
# ============================================================
# Cell 5: Load all 6 models (OV + PyTorch) — defensive print
# ============================================================
core = ov.Core()
MODELS = {}        # name → loaded model object (compiled OV or PyTorch nn.Module)
MODEL_FORMAT = {}
TIME_FRAMES = {}
OV_INPUTS = {}     # name → input port (for OV models)

def safe_print_io(compiled):
    """Print I/O info without crashing on missing names or dynamic shapes."""
    try:
        inputs = list(compiled.inputs)
        if inputs:
            try: name = inputs[0].any_name
            except: name = '?'
            try: shape = str(inputs[0].partial_shape)
            except: shape = '?'
            print(f'  input: {name} shape={shape}')
    except Exception as e:
        print(f'  (input info unavailable: {type(e).__name__})')
    try:
        outputs = list(compiled.outputs)
        for o in outputs:
            try: name = o.any_name
            except: name = '?'
            try: shape = str(o.partial_shape)
            except: shape = '?'
            print(f'  output: {name} shape={shape}')
    except Exception as e:
        print(f'  (output info unavailable: {type(e).__name__})')


for mname, cfg in MODELS_CFG.items():
    print(f'\n--- Loading {mname} ---')
    DATASET_ROOTS = [
        Path('/kaggle/input/datasets/maekeso') / cfg['dataset_slug'],
        Path('/kaggle/input') / cfg['dataset_slug'],
    ]
    ds_root = None
    for p in DATASET_ROOTS:
        if p.exists():
            ds_root = p; break
    assert ds_root is not None, f'{cfg["dataset_slug"]} not mounted'

    mel_cfg = MEL_CONFIGS[cfg['mel']]
    duration = cfg['duration']
    n_samples = SR * duration
    time_frames = n_samples // mel_cfg['hop'] + 1
    TIME_FRAMES[mname] = time_frames
    print(f'  mel: {cfg["mel"]} ({mel_cfg["n_mels"]}, {time_frames})')

    if cfg['format'] == 'ov':
        ov_path = ds_root / cfg['ov_path']
        assert ov_path.exists(), f'OV not found: {ov_path}'
        ov_model = core.read_model(str(ov_path))
        compiled = core.compile_model(ov_model, 'CPU')
        MODELS[mname] = compiled
        MODEL_FORMAT[mname] = 'ov'
        # Use input PORT directly (bypass name lookup)
        OV_INPUTS[mname] = compiled.input(0)
        print(f'  format: OV')
        safe_print_io(compiled)
    else:
        # PyTorch ckpt
        ckpt_path = ds_root / cfg['pytorch_ckpt']
        if not ckpt_path.exists():
            for f in ds_root.rglob(Path(cfg['pytorch_ckpt']).name):
                ckpt_path = f; break
        assert ckpt_path.exists(), f'ckpt not found in {ds_root}'
        state = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
        model = BirdSEDModel(
            backbone_name=cfg['backbone'],
            n_mels=mel_cfg['n_mels'],
            time_frames=time_frames,
        ).eval()
        msg = model.load_state_dict(state['model_state'], strict=False)
        MODELS[mname] = model
        MODEL_FORMAT[mname] = 'pytorch'
        print(f'  format: PyTorch CPU')
        print(f'  ckpt: {ckpt_path.name}, params={sum(p.numel() for p in model.parameters())/1e6:.1f}M')
        print(f'  load: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}')

print(f'\n=== Loaded {len(MODELS)} models ===')


In [ ]:
# ============================================================
# Cell 6: Inference helpers — robust to OV models without output names
# ============================================================
GAUSSIAN_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1], dtype=np.float32)
N_OUT_ROWS = 12

def load_60s(path):
    """Load 60s mono 32kHz audio."""
    try:
        wav, sr = sf.read(str(path), dtype='float32', always_2d=False)
    except Exception:
        return np.zeros(SR * 60, dtype=np.float32)
    if wav.ndim > 1: wav = wav.mean(axis=1)
    if sr != SR: wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    target = SR * 60
    if len(wav) < target: wav = np.pad(wav, (0, target - len(wav)))
    elif len(wav) > target: wav = wav[:target]
    return wav.astype(np.float32)


def prepare_input_simple(audio_60s, duration):
    """5s simple chunking: (12, duration*SR)."""
    n_samples = SR * duration
    return audio_60s.reshape(N_OUT_ROWS, n_samples)


def prepare_input_sliding(audio_60s, duration):
    """Sliding window for 10s/20s: padded + 12 windows step 5s.

    ★ Padding spec (working v6 pattern):
    - padded_sec = (12-1)*5 + duration = 65 (10s) or 75 (20s)
    - pad_lead_samples = (padded_sec - 60) * SR // 2  ← samples-based (0.5s 切捨て無)
    - 10s case: 80000 samples (2.5s) lead, 80000 trail
    - 20s case: 240000 samples (7.5s) lead, 240000 trail
    """
    padded_sec = (N_OUT_ROWS - 1) * 5 + duration   # 65 (10s), 75 (20s)
    padded_samples = padded_sec * SR
    # ★ Samples-based division to preserve 0.5s precision (working v6 pattern)
    pad_lead_samples = ((padded_sec - 60) * SR) // 2
    audio_padded = np.zeros(padded_samples, dtype=np.float32)
    audio_padded[pad_lead_samples:pad_lead_samples + len(audio_60s)] = audio_60s
    chunks = np.zeros((N_OUT_ROWS, SR * duration), dtype=np.float32)
    step_samples = 5 * SR
    win_samples = duration * SR
    for k in range(N_OUT_ROWS):
        start = k * step_samples
        chunks[k] = audio_padded[start:start + win_samples]
    return chunks


def identify_ov_outputs(result_dict):
    """Identify clip_logits (2D) and framewise (3D) from OV result by SHAPE.
    Robust to missing output names."""
    arrays = list(result_dict.values())
    clip_logits = None
    framewise = None
    for arr in arrays:
        if arr.ndim == 2:
            clip_logits = arr
        elif arr.ndim == 3:
            framewise = arr
    if clip_logits is None or framewise is None:
        raise RuntimeError(f'Could not identify outputs by shape. Got: {[a.shape for a in arrays]}')
    return clip_logits, framewise


def infer_model(mname, chunks_np):
    """Forward one model on chunks, return blend_logits (12, 234)."""
    cfg = MODELS_CFG[mname]
    mel_cfg = MEL_CONFIGS[cfg['mel']]
    mel_tf = MEL_TRANSFORMS[cfg['mel']]
    model = MODELS[mname]
    fmt = MODEL_FORMAT[mname]

    # Mel computation
    wav_t = torch.from_numpy(chunks_np).unsqueeze(1)   # (12, 1, samples)
    mel = mel_tf(wav_t)
    mel = normalize_mel(mel)

    if fmt == 'ov':
        # OV: use input PORT (bypass name lookup)
        mel_np = mel.numpy().astype(np.float32)
        input_port = OV_INPUTS[mname]
        result = model.create_infer_request().infer({input_port: mel_np})
        clip_logits, framewise = identify_ov_outputs(result)
        # framewise shape: (B, time, num_class) — max over time = axis 1
        frame_max = framewise.max(axis=1)
        blend_logits = 0.5 * clip_logits + 0.5 * frame_max
    else:
        # PyTorch
        with torch.no_grad():
            clip_logits, framewise = model(mel, return_framewise=True)
            # framewise shape: (B, time, num_class) after permute in model
            frame_max = framewise.max(dim=1).values
            blend_logits = 0.5 * clip_logits + 0.5 * frame_max
            blend_logits = blend_logits.float().cpu().numpy()

    return blend_logits.astype(np.float32)   # (12, 234)


def sigmoid_np(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)


print('OK inference helpers ready (PORT-based + shape identify)')

# ============================================================
# Smoke test: 1 model 1 file で動作確認
# ============================================================
print('\n=== Smoke test (1 file × 1 model) ===')
if MODELS_CFG:
    test_mname = list(MODELS_CFG.keys())[0]
    cfg = MODELS_CFG[test_mname]
    # Dummy 60s audio
    dummy_audio = np.random.randn(SR * 60).astype(np.float32) * 0.01
    chunks = prepare_input_simple(dummy_audio, cfg['duration']) if cfg['window_type'] == 'simple' else prepare_input_sliding(dummy_audio, cfg['duration'])
    print(f'  Test model: {test_mname} ({cfg["window_type"]}, {cfg["duration"]}s)')
    print(f'  chunks shape: {chunks.shape}')
    try:
        out = infer_model(test_mname, chunks)
        print(f'  output shape: {out.shape}, dtype={out.dtype}')
        print(f'  output stats: mean={out.mean():.4f}, std={out.std():.4f}, max={out.max():.4f}')
        print(f'  [OK] Smoke test passed')
    except Exception as e:
        print(f'  [FAIL] {type(e).__name__}: {str(e)[:300]}')
        raise


In [ ]:
import glob
# ============================================================
# Cell 7: Main inference loop with speed measurement
# ============================================================
test_files = sorted(glob.glob(str(TEST_DIR / '*.ogg'))) if TEST_DIR.is_dir() else []
if len(test_files) == 0:
    fallback = BASE / 'train_soundscapes'
    if fallback.is_dir():
        test_files = sorted(glob.glob(str(fallback / '*.ogg')))[:5]
        print(f'No test_soundscapes, using {len(test_files)} train files for debug')
print(f'Test files: {len(test_files)}')

# Per-model timing tracker
model_times = {mname: 0.0 for mname in MODELS_CFG}

all_rows = []
all_blended_logits = []
t_inference_start = time.time()

for fi, fp in enumerate(test_files):
    fp = Path(fp)
    audio = load_60s(fp)

    # Per-model inference, blend in logit space
    blended_logits = np.zeros((N_OUT_ROWS, N_CLASSES), dtype=np.float32)
    for mname, cfg in MODELS_CFG.items():
        t0 = time.time()
        if cfg['window_type'] == 'simple':
            chunks = prepare_input_simple(audio, cfg['duration'])
        else:
            chunks = prepare_input_sliding(audio, cfg['duration'])
        logits = infer_model(mname, chunks)
        blended_logits += cfg['weight'] * logits
        model_times[mname] += time.time() - t0

    # Save per-file row_ids + logits
    stem = fp.stem
    for k in range(N_OUT_ROWS):
        all_rows.append(f'{stem}_{(k+1)*5}')
    all_blended_logits.append(blended_logits)

    # Progress
    if (fi + 1) % 50 == 0 or fi == 0 or fi == len(test_files) - 1:
        elapsed = time.time() - t_inference_start
        rate = (fi + 1) / max(elapsed, 1e-6)
        eta_min = (len(test_files) - fi - 1) / max(rate, 1e-6) / 60
        print(f'  [{fi+1:4d}/{len(test_files)}] {elapsed:.0f}s {rate:.2f}f/s eta={eta_min:.1f}min')

# Concatenate logits across files
logits_arr = np.concatenate(all_blended_logits, axis=0).astype(np.float32)   # (n_files*12, 234)
print(f'\nInference DONE: {len(all_rows)} rows in {(time.time()-t_inference_start)/60:.1f} min')
print(f'\nPer-model timing (total seconds):')
for mname, total in sorted(model_times.items(), key=lambda x: -x[1]):
    pct = 100 * total / sum(model_times.values())
    print(f'  {mname:15s} {total:6.1f}s ({pct:5.1f}%)')


In [ ]:
# ============================================================
# Cell 8: Logit smoothing + sigmoid + submission CSV
# ============================================================
# Smooth in logit space across 12 windows per file
def smooth_per_file(arr, kernel=GAUSSIAN_KERNEL):
    smoothed = arr.reshape(-1, N_OUT_ROWS, arr.shape[1]).copy()
    for i in range(smoothed.shape[0]):
        smoothed[i] = convolve1d(smoothed[i], kernel, axis=0, mode='nearest')
    return smoothed.reshape(-1, arr.shape[1])

logits_smoothed = smooth_per_file(logits_arr)
probs = sigmoid_np(logits_smoothed)

# Build submission CSV
sub = pd.DataFrame(probs, columns=PRIMARY_LABELS)
sub.insert(0, 'row_id', all_rows)
assert sub['row_id'].nunique() == len(sub), 'duplicate row_id'
print(f'sub_df: {sub.shape}, mean={sub[PRIMARY_LABELS].mean().mean():.4f}, max={sub[PRIMARY_LABELS].max().max():.4f}')

# Reindex to match sample_sub order
if len(test_files) > 0 and len(SAMPLE_SUB) > 0:
    sub = sub.set_index('row_id').reindex(SAMPLE_SUB['row_id']).reset_index()
    sub[PRIMARY_LABELS] = sub[PRIMARY_LABELS].fillna(0.0)

sub_path = OUT_DIR / 'submission.csv'
sub.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({sub_path.stat().st_size/1e6:.1f}MB)')
print(sub.head(3))

total_time = time.time() - GLOBAL_START
print(f'\n=== TOTAL TIME: {total_time/60:.1f} min ===')
